# sum-and-broadcast-duality — ex2: check_adjoint: verify sum_back is the adjoint of sum via <Av, w> = <v, A^T w>

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sum-and-broadcast-duality`. Running the final beacon cell reports progress against the `Backprop: sum/broadcast duality` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum/broadcast duality` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-and-broadcast-duality`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-and-broadcast-duality"
DD_SUBTOPIC = "Backprop: sum/broadcast duality"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## sum/broadcast duality — the adjoint identity — quick refresher

A linear map `A` has an adjoint `A^T` (transpose for real-valued tensors) characterized by the **inner-product identity**:

$$\langle A v, w \rangle = \langle v, A^T w \rangle$$

For backprop: the forward op is `A`, the back-fn is `A^T`. So:

- forward: `sum(x, dim=k)` ≡ multiplying by a row-vector of 1s along axis k. Linear map A.
- backward: `sum_back(g, ..., dim=k)` ≡ broadcasting g back along axis k. Linear map A^T.

The identity becomes: for any inputs x (shape of forward arg) and y (shape of forward output),

$$\langle \mathrm{sum}(x, k), y \rangle = \langle x, \mathrm{sum\_back}(y, k) \rangle$$

If `sum_back` is truly the adjoint, this equality holds for every x, y. If it isn't, the identity will fail on a random probe — and gradients downstream would be silently wrong.

### Exercise 2 — check_adjoint: verify sum_back is the adjoint of sum via <Av, w> = <v, A^T w>

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the adjoint identity `<Av, w> == <v, A^T w>` to verify operationally that sum_back is the transpose of sum as a linear map — the deeper meaning of 'duality' beyond shape matching.
> Keywords: adjoint, transpose, inner-product, duality, sum-back, linear-map
> ```

**KCs targeted:** `sum-and-broadcast-duality`, `unbroadcast-pattern`

Implement two functions:

**1. `sum_back(grad_out, x, dim)`** — same as ex1's `sum_back` with `keepdim=False`. Unsqueeze grad_out at `dim`, then `.expand_as(x).clone()`.

**2. `check_adjoint(x, dim)`** — verify the adjoint identity. Given an input shape (via concrete tensor `x`) and a reduction axis `dim`:

1. Compute `Ax = x.sum(dim=dim)` (shape: x.shape with dim removed).
2. Sample a random `y` with the same shape as `Ax`.
3. Compute LHS = `(Ax * y).sum()` — the inner product `<Ax, y>`.
4. Compute RHS = `(x * sum_back(y, x, dim)).sum()` — the inner product `<x, A^T y>`.
5. Return `t.allclose(LHS, RHS, atol=1e-5)`.

If `sum_back` is correctly the adjoint, the function returns True for every choice of `x.shape` and `dim`. If `sum_back` is buggy (e.g. forgets `unsqueeze`, uses wrong dim, applies `keepdim=True` semantics), the identity FAILS — visible as a scalar mismatch.

Use `t.manual_seed(0)` before sampling `y` so the test is deterministic.

**Why this matters more than shape tests.** Ex1's tests checked `g.shape == x.shape` and a few hand-computed values. Shape can be right but values wrong — e.g. if you forget `unsqueeze` and accidentally broadcast across the wrong axis. The adjoint identity is a SCALAR check that catches any deviation from true linear-algebraic transpose-ness.

In [ ]:
def sum_back(grad_out: Tensor, x: Tensor, dim: int) -> Tensor:
    """Backward of x.sum(dim, keepdim=False). Broadcast grad_out to x.shape."""
    raise NotImplementedError()


def check_adjoint(x: Tensor, dim: int) -> bool:
    """Verify <Ax, y> == <x, A^T y> where A = sum(dim), A^T = sum_back(dim)."""
    raise NotImplementedError()


def _test_ex2():
    # --- sum_back basic shape ---
    x = t.arange(12, dtype=t.float32).reshape(3, 4)
    g = sum_back(t.ones(3), x, dim=1)
    assert g.shape == (3, 4), f'shape: {g.shape}'
    assert t.allclose(g, t.ones(3, 4))

    # --- adjoint identity holds for 2-D reductions ---
    t.manual_seed(0)
    x = t.randn(3, 4)
    assert check_adjoint(x, dim=0) is True, '<Ax, y> != <x, A^T y> on dim=0'
    assert check_adjoint(x, dim=1) is True, '<Ax, y> != <x, A^T y> on dim=1'

    # --- adjoint identity holds for 3-D reductions across every axis ---
    x = t.randn(2, 3, 4)
    for dim in range(3):
        assert check_adjoint(x, dim=dim) is True, (
            f'adjoint identity must hold for dim={dim} on 3-D tensor'
        )

    # --- adjoint identity holds for various shapes ---
    for shape in [(5,), (5, 6), (2, 3, 4), (1, 7, 2)]:
        x = t.randn(shape)
        for dim in range(len(shape)):
            ok = check_adjoint(x, dim=dim)
            assert ok, f'adjoint failed: shape={shape}, dim={dim}'

    # --- explicit LHS == RHS via hand-computed inner products ---
    # This is the underlying identity, broken out so the test author can
    # read what 'adjoint' actually means.
    x = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)
    Ax = x.sum(dim=1)                                 # shape (2,) = [6, 15]
    y = t.tensor([10.0, 100.0])                       # shape (2,)
    lhs = (Ax * y).sum()                              # 6*10 + 15*100 = 60 + 1500 = 1560
    rhs = (x * sum_back(y, x, dim=1)).sum()
    assert t.allclose(lhs, rhs, atol=1e-5), (
        f'hand-computed adjoint identity: lhs={lhs.item()}, rhs={rhs.item()}'
    )
    assert lhs.item() == 1560.0

    # --- check_adjoint is deterministic (same x, dim → same result) ---
    x = t.randn(4, 5)
    r1 = check_adjoint(x, dim=0)
    r2 = check_adjoint(x, dim=0)
    assert r1 == r2, 'check_adjoint must be deterministic given x'

    # --- sanity: sum_back agrees with torch.autograd on a small case ---
    x_ref = t.tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
    y_ref = x_ref.sum(dim=1).sum()
    y_ref.backward()
    g_ours = sum_back(t.ones(2), x_ref.detach(), dim=1)
    assert t.allclose(g_ours, x_ref.grad), (
        f'sum_back must match autograd: ours={g_ours}, ref={x_ref.grad}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def sum_back(grad_out: Tensor, x: Tensor, dim: int) -> Tensor:
    # keepdim=False semantics: re-insert the dropped axis, then expand.
    grad_out = grad_out.unsqueeze(dim)
    return grad_out.expand_as(x).clone()


def check_adjoint(x: Tensor, dim: int) -> bool:
    # Use a fixed seed for reproducibility — same x/dim → same y.
    gen = t.Generator().manual_seed(0)
    Ax = x.sum(dim=dim)
    y = t.randn(Ax.shape, generator=gen)
    lhs = (Ax * y).sum()                         # <Ax, y>
    rhs = (x * sum_back(y, x, dim=dim)).sum()    # <x, A^T y>
    return bool(t.allclose(lhs, rhs, atol=1e-5))
```

**Why the adjoint identity is the gold standard.** Shape-only tests can pass while semantics are wrong (e.g. using `expand_as` without `unsqueeze` first — produces the right shape with completely wrong stride pattern on broadcasting edge cases). The inner-product identity is a SCALAR equality that captures the linear-algebraic relationship exactly.

**`<Ax, y> = <x, A^T y>` IS the definition of adjoint.** Every back-fn in autograd should pass this test against its corresponding forward. Frameworks like JAX even use it for automated gradient checking (`jax.test_util.check_grads`).

**Why `Generator` not global `manual_seed`.** Calling `t.manual_seed(0)` would mutate global state, affecting anything else in the test. A local `Generator` keeps the randomness reproducible AND scoped.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()